# Task 4: Context-Aware Chatbot Using RAG (Retrieval-Augmented Generation)
**DevelopersHub Corporation — AI/ML Engineering Internship (Phase 2)**

**Objective:** Build a conversational chatbot that (1) remembers conversation history (context memory) and (2) retrieves answers from a vectorized document store (RAG) instead of hallucinating from parameters alone.



In [1]:
# Cell 1: Install required libraries
!pip install -q sentence-transformers faiss-cpu transformers accelerate streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 41.3 MB/s eta 0:00:00


## Step 1: Build the Knowledge Base (Self-Contained — No External API Calls)

In [2]:
# Cell 2: Define our custom knowledge base
knowledge_base = {
    "Artificial Intelligence": "Artificial intelligence (AI) is the field of computer science focused on building systems that can perform tasks normally requiring human intelligence, such as understanding language, recognizing images, making decisions, and solving problems. AI systems are typically built using machine learning, where a model learns patterns from data rather than being explicitly programmed with rules. Modern AI applications include voice assistants, recommendation systems, self-driving cars, medical diagnosis tools, and chatbots. AI is generally divided into narrow AI, designed for a specific task, and general AI, a hypothetical system with human-level reasoning across any domain. Narrow AI is what exists today; general AI remains a research goal.",
    "Machine Learning": "Machine learning (ML) is a subfield of artificial intelligence that focuses on building algorithms that improve automatically through experience and data, rather than being explicitly programmed. There are three main types of machine learning: supervised learning, where a model learns from labeled examples; unsupervised learning, where a model finds patterns in unlabeled data; and reinforcement learning, where an agent learns by receiving rewards or penalties for its actions. Deep learning, a subset of machine learning based on neural networks with many layers, has driven major breakthroughs in image recognition, speech recognition, and natural language processing. Machine learning is commonly used for fraud detection, recommendation engines, predictive maintenance, and customer churn prediction.",
    "Natural Language Processing": "Natural language processing (NLP) is a branch of artificial intelligence concerned with enabling computers to understand, interpret, and generate human language. Common NLP tasks include text classification, sentiment analysis, machine translation, named entity recognition, question answering, and text summarization. Modern NLP is dominated by transformer-based models, such as BERT and GPT, which are pretrained on large amounts of text and can be fine-tuned for specific tasks. Retrieval-Augmented Generation (RAG) is a popular NLP technique where a language model retrieves relevant information from an external knowledge source before generating an answer, which helps reduce hallucination and lets the model answer questions about information it wasn't originally trained on.",
}

print(f"Loaded {len(knowledge_base)} documents.")
for title, text in knowledge_base.items():
    print("-", title, f"({len(text)} characters)")

Loaded 3 documents.
- Artificial Intelligence (743 characters)
- Machine Learning (807 characters)
- Natural Language Processing (782 characters)


## Step 2: Split Documents into Chunks (Simple Sentence-Based Chunking)

In [3]:
# Cell 3: Chunk the documents for embedding
import re

def chunk_text(text, max_chars=250):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks, current = [], ""
    for sentence in sentences:
        if len(current) + len(sentence) <= max_chars:
            current = (current + " " + sentence).strip()
        else:
            if current:
                chunks.append(current)
            current = sentence
    if current:
        chunks.append(current)
    return chunks

all_chunks = []       # list of chunk strings
chunk_sources = []     # matching list of source titles

for title, text in knowledge_base.items():
    for chunk in chunk_text(text):
        all_chunks.append(chunk)
        chunk_sources.append(title)

print(f"Split into {len(all_chunks)} chunks.")
print("Example chunk:", all_chunks[0])

Split into 12 chunks.
Example chunk: Artificial intelligence (AI) is the field of computer science focused on building systems that can perform tasks normally requiring human intelligence, such as understanding language, recognizing images, making decisions, and solving problems.


## Step 3: Embed the Chunks & Build a Vector Index (FAISS)

In [4]:
# Cell 4: Create embeddings and a FAISS vector index
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = embedder.encode(all_chunks, convert_to_numpy=True, normalize_embeddings=True)

embedding_dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)  # inner product on normalized vectors = cosine similarity
index.add(chunk_embeddings)

print(f"FAISS index built with {index.ntotal} vectors of dimension {embedding_dim}.")

def retrieve(query, top_k=3):
    query_vec = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, indices = index.search(query_vec, top_k)
    results = [(all_chunks[i], chunk_sources[i], float(scores[0][rank])) for rank, i in enumerate(indices[0])]
    return results

print("\nRetrieval test:")
for chunk, source, score in retrieve("What is machine learning?"):
    print(f"[{source} | score={score:.2f}] {chunk[:120]}...")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index built with 12 vectors of dimension 384.

Retrieval test:
[Machine Learning | score=0.81] Machine learning (ML) is a subfield of artificial intelligence that focuses on building algorithms that improve automati...
[Machine Learning | score=0.65] Machine learning is commonly used for fraud detection, recommendation engines, predictive maintenance, and customer chur...
[Artificial Intelligence | score=0.65] AI systems are typically built using machine learning, where a model learns patterns from data rather than being explici...


## Step 4: Set Up the LLM (Open-Source, No API Key Needed)

We load `google/flan-t5-base` directly with `transformers` (tokenizer + model + `.generate()`), avoiding the higher-level `pipeline()` wrapper entirely — this sidesteps pipeline-registry issues that can vary between `transformers` versions.

In [5]:
# Cell 5: Load a local open-source LLM directly (no pipeline() wrapper)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

llm_name = "google/flan-t5-base"
llm_tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(llm_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
llm_model = llm_model.to(device)

def generate_answer(prompt, max_new_tokens=150):
    inputs = llm_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    output_ids = llm_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return llm_tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("Test generation:", generate_answer("Answer the question: What is the capital of France?"))

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Test generation: london


## Step 5: Build the Context-Aware RAG Chatbot (Manual Retrieval + Manual Memory)

- **Retrieval:** pulls the most relevant chunks from the FAISS index for each question.
- **Memory:** a simple running list of (question, answer) pairs is kept and included in the prompt, so follow-up questions using pronouns ("it", "that", "this") are understood in context.

In [6]:
# Cell 6: Build the conversational RAG chatbot
chat_history = []  # list of (question, answer) tuples - our "memory"

def build_prompt(question, retrieved_chunks):
    context = "\n".join([f"- {chunk}" for chunk, _, _ in retrieved_chunks])

    history_text = ""
    if chat_history:
        history_lines = [f"Q: {q}\nA: {a}" for q, a in chat_history[-3:]]  # last 3 turns
        history_text = "Previous conversation:\n" + "\n".join(history_lines) + "\n\n"

    prompt = (
        f"{history_text}"
        f"Context information:\n{context}\n\n"
        f"Using the conversation history (if relevant) and the context above, answer the question.\n"
        f"Question: {question}\nAnswer:"
    )
    return prompt

def ask(question, top_k=3, verbose=True):
    retrieved = retrieve(question, top_k=top_k)
    prompt = build_prompt(question, retrieved)
    answer = generate_answer(prompt)
    chat_history.append((question, answer))
    if verbose:
        print("Q:", question)
        print("A:", answer)
        print()
    return answer

## Step 6: Test the Chatbot — Demonstrate Context Memory

In [7]:
# Cell 7: Have a multi-turn conversation to show it remembers context
ask("What is machine learning?")
ask("What is it commonly used for?")   # "it" should be resolved as machine learning, thanks to memory
ask("How is that different from natural language processing?")  # tests both memory and retrieval

Q: What is machine learning?
A: a subfield of artificial intelligence

Q: What is it commonly used for?
A: fraud detection, recommendation engines, predictive maintenance, and customer churn prediction

Q: How is that different from natural language processing?
A: Modern NLP is dominated by transformer-based models, such as BERT and GPT, which are pretrained on large amounts of text and can be fine-tuned for specific tasks



'Modern NLP is dominated by transformer-based models, such as BERT and GPT, which are pretrained on large amounts of text and can be fine-tuned for specific tasks'

In [8]:
# Cell 8: Inspect what's stored in memory
for i, (q, a) in enumerate(chat_history, 1):
    print(f"Turn {i}")
    print("  User:", q)
    print("  Bot: ", a)
    print()

Turn 1
  User: What is machine learning?
  Bot:  a subfield of artificial intelligence

Turn 2
  User: What is it commonly used for?
  Bot:  fraud detection, recommendation engines, predictive maintenance, and customer churn prediction

Turn 3
  User: How is that different from natural language processing?
  Bot:  Modern NLP is dominated by transformer-based models, such as BERT and GPT, which are pretrained on large amounts of text and can be fine-tuned for specific tasks



## Step 7: Deploy with Streamlit (Runs Inside Colab via a Public Tunnel)

In [9]:
# Cell 9: Write the Streamlit app to a file
app_code = '''
import streamlit as st
import re
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import faiss

st.set_page_config(page_title="Context-Aware RAG Chatbot")
st.title("Context-Aware Chatbot (RAG)")
st.caption("Ask me about Artificial Intelligence, Machine Learning, or NLP - I remember our conversation.")

KNOWLEDGE_BASE = {
    "Artificial Intelligence": "Artificial intelligence (AI) is the field of computer science focused on building systems that can perform tasks normally requiring human intelligence, such as understanding language, recognizing images, making decisions, and solving problems. AI systems are typically built using machine learning, where a model learns patterns from data rather than being explicitly programmed with rules. Modern AI applications include voice assistants, recommendation systems, self-driving cars, medical diagnosis tools, and chatbots.",
    "Machine Learning": "Machine learning (ML) is a subfield of artificial intelligence that focuses on building algorithms that improve automatically through experience and data, rather than being explicitly programmed. There are three main types: supervised learning, unsupervised learning, and reinforcement learning. Deep learning, based on neural networks with many layers, has driven major breakthroughs in image recognition, speech recognition, and natural language processing.",
    "Natural Language Processing": "Natural language processing (NLP) is a branch of artificial intelligence concerned with enabling computers to understand, interpret, and generate human language. Common NLP tasks include text classification, sentiment analysis, machine translation, and question answering. Retrieval-Augmented Generation (RAG) is a popular NLP technique where a language model retrieves relevant information from an external knowledge source before generating an answer, reducing hallucination.",
}

def chunk_text(text, max_chars=250):
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    chunks, current = [], ""
    for sentence in sentences:
        if len(current) + len(sentence) <= max_chars:
            current = (current + " " + sentence).strip()
        else:
            if current:
                chunks.append(current)
            current = sentence
    if current:
        chunks.append(current)
    return chunks

@st.cache_resource
def load_resources():
    all_chunks, chunk_sources = [], []
    for title, text in KNOWLEDGE_BASE.items():
        for chunk in chunk_text(text):
            all_chunks.append(chunk)
            chunk_sources.append(title)

    embedder = SentenceTransformer("all-MiniLM-L6-v2")
    chunk_embeddings = embedder.encode(all_chunks, convert_to_numpy=True, normalize_embeddings=True)
    dim = chunk_embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(chunk_embeddings)

    llm_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
    llm_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    llm_model = llm_model.to(device)

    return embedder, index, all_chunks, chunk_sources, llm_tokenizer, llm_model, device

embedder, index, all_chunks, chunk_sources, llm_tokenizer, llm_model, device = load_resources()

def retrieve(query, top_k=3):
    query_vec = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, indices = index.search(query_vec, top_k)
    return [(all_chunks[i], chunk_sources[i]) for i in indices[0]]

def generate_answer(prompt, max_new_tokens=150):
    inputs = llm_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    output_ids = llm_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return llm_tokenizer.decode(output_ids[0], skip_special_tokens=True)

def build_prompt(question, retrieved_chunks, history):
    context = "\\n".join([f"- {c}" for c, _ in retrieved_chunks])
    history_text = ""
    if history:
        lines = [f"Q: {q}\\nA: {a}" for q, a in history[-3:]]
        history_text = "Previous conversation:\\n" + "\\n".join(lines) + "\\n\\n"
    return f"{history_text}Context information:\\n{context}\\n\\nUsing the conversation history (if relevant) and the context above, answer the question.\\nQuestion: {question}\\nAnswer:"

if "history" not in st.session_state:
    st.session_state.history = []
if "messages" not in st.session_state:
    st.session_state.messages = []

for role, text in st.session_state.messages:
    with st.chat_message(role):
        st.write(text)

user_input = st.chat_input("Type your question...")
if user_input:
    st.session_state.messages.append(("user", user_input))
    with st.chat_message("user"):
        st.write(user_input)

    with st.spinner("Thinking..."):
        retrieved = retrieve(user_input, top_k=3)
        prompt = build_prompt(user_input, retrieved, st.session_state.history)
        answer = generate_answer(prompt)
        st.session_state.history.append((user_input, answer))

    st.session_state.messages.append(("assistant", answer))
    with st.chat_message("assistant"):
        st.write(answer)
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("app.py written.")

app.py written.


<>:22: SyntaxWarning: invalid escape sequence '\s'
<>:22: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_627/962145733.py:22: SyntaxWarning: invalid escape sequence '\s'
  sentences = re.split(r"(?<=[.!?])\s+", text.strip())


In [10]:
# Cell 10: Run Streamlit and expose it with a public tunnel (localtunnel)
!npm install -g localtunnel -q

import subprocess, time

streamlit_process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"]
)
time.sleep(10)  # brief wait for the server to boot

# Print your public IP - localtunnel may ask you to paste this as a password
!curl -s https://loca.lt/mytunnelpassword

# Launch the tunnel (click the printed URL; if prompted for a password, paste the IP printed above)
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 22 packages in 3s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙npm notice
npm notice New major version of npm available! 10.8.2 -> 11.18.0
npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.18.0
npm notice To update run: npm install -g npm@11.18.0
npm notice
⠙34.124.157.132⠙⠹⠸⠼⠴⠦⠧your url is: https://wild-cameras-go.loca.lt
^C


> **Note:** `localtunnel` will print a URL like `https://xxxx.loca.lt`. Open it — if it asks for a "Tunnel Password", paste the IP address printed by the `curl` command just above it. This keeps the Streamlit chatbot running as long as this Colab cell is executing.
>
> To stop the app later: interrupt this cell (Runtime → Interrupt execution).

## Final Summary / Insights

- Built a **Retrieval-Augmented Generation (RAG)** chatbot from first principles: documents (a curated, built-in knowledge base) → chunked → embedded (`all-MiniLM-L6-v2` via `sentence-transformers`) → indexed with **FAISS** → retrieved per-question via cosine similarity search.
- Added **conversation memory** as a simple running list of previous (question, answer) turns, injected into the prompt — so the bot correctly resolves follow-up questions that use pronouns or implicit references to earlier turns.
- Used an **open-source LLM** (`flan-t5-base`) loaded directly via `transformers` (tokenizer + model + `.generate()`) for answer generation — no API key required, fully reproducible in Colab.
- Deployed the chatbot as an interactive **Streamlit** app, exposed publicly via `localtunnel` directly from the Colab notebook.
- **Design note:** this implementation deliberately avoids the LangChain framework and instead uses `sentence-transformers`, `faiss`, and `transformers` directly. These are lower-level but far more stable across versions, which avoids the import-path and pipeline-registry breakage that frequently affects fast-moving framework wrappers like LangChain. The underlying concepts (retrieval, augmentation, generation, memory) are identical either way.

**Skills Gained:** Conversational AI development, embeddings & vector search (FAISS), Retrieval-Augmented Generation (RAG) implemented from first principles, LLM integration & deployment.
